In [354]:
import sympy as sym
from sympy import pretty, det, diff
from sympy.physics.mechanics import dynamicsymbols, init_vprinting
from sympy.printing.latex import latex
from IPython.display import display, Markdown

init_vprinting()

## Basic Methods

In [355]:
"""
This function is used if latex rendering is not available
"""
def label_print(matrix, label="M"):
    """Print matrix with label in plain text format"""
    matrix_str = pretty(matrix)
    lines = matrix_str.split('\n')
    label_text = f"{label} = "
    label_width = len(label_text)

    mid_index = len(lines) // 2  # Middle row index for centering

    formatted = '\n'.join(
        [
            (label_text if i == mid_index else ' ' * label_width) + line
            for i, line in enumerate(lines)
        ]
    )

    print(formatted)

"""
This function is used to render the derivative with the dot notation 
"""
def latex_with_dot(expr):
    def format_derivative(d):
        if isinstance(d, sym.Derivative):
            arg = d.expr
            if isinstance(arg, sym.Function):
                fname = arg.func.__name__
                order = len(d.variables)
                dot_map = {1: r"\dot", 2: r"\ddot"}
                if order in dot_map:
                    return r"{}{{{}}}\left({}\right)".format(dot_map[order], fname, latex(d.variables[0]))
                else:
                    return r"\overset{{({})}}{{{}}}\left({}\right)".format(order, fname, latex(d.variables[0]))
        return latex(d)

    # If input is a matrix, apply formatting element-wise
    if isinstance(expr, sym.Matrix):
        rows, cols = expr.shape
        return r"\begin{bmatrix}" + \
               r" \\".join([
                   " & ".join([format_derivative(expr[i, j]) for j in range(cols)])
                   for i in range(rows)
               ]) + r"\end{bmatrix}"
    else:
        return format_derivative(expr)


## Notations and Definitions

In [356]:
"""
    Definitions of basic symbols
"""
# Define dynamic symbols
x = dynamicsymbols('x')
y = dynamicsymbols('y')
z = dynamicsymbols('Z')

# Define symbols for pixel velocities
s_dot = sym.Matrix([                                                                                                      
    [x.diff()],                                                                                                           
    [y.diff()]                                                                                                            
])                                                                                                                        

# Zero matrix for block construction
O3 = sym.zeros(3,3)

string = (
    f'#### x coordinate in pixel space: ${latex(x)}$  \n'
    f'#### y coordinate in pixel space: ${latex(y)}$  \n '
    f'#### Depth Z of the (x, y) point: ${latex(z)}$  \n '
    f'#### Pixel velocity matrix $\dot{{s}}$ = ${latex_with_dot(s_dot)}$'
)

display(Markdown(string))

#### x coordinate in pixel space: $x{\left(t \right)}$  
#### y coordinate in pixel space: $y{\left(t \right)}$  
 #### Depth Z of the (x, y) point: $Z{\left(t \right)}$  
 #### Pixel velocity matrix $\dot{s}$ = $\begin{bmatrix}\dot{x}\left(t\right) \\\dot{y}\left(t\right)\end{bmatrix}$

In [357]:
"""
    Definition of the rotation and postion matrices between robot frame and camera frame
"""
# Define position matrix form robot base frame to camera frame
Prc = sym.Matrix([
    [0],
    [0],
    [0] 
])

# Define rotation matrix from robot base frame to camera frame
Rrc = sym.Matrix([
    [0, 0, 1],
    [-1, 0, 0],
    [0, -1, 0]
])

Trc = sym.zeros(4,4)        # Homogeneous Transformation from robot base to camera 
Trc[:3, :3] = Rrc
Trc[:3, 3] = Prc

Tcr = sym.zeros(4, 4)       # Homogeneous Transformation from camera to robot base
Tcr[:3, :3] = Trc[:3,:3].T 
Tcr[:3, 3] = -Trc[:3, :3] * Trc[:3, 3]

string = (
    f'### Position between Camera  and Robot:\n'
    f'$\\large p_{{rc}}$ = $\\Large{latex(Prc)}$\n'
    f'### Orientation between Camera  and Robot:\n'
    f'$\\large R_{{rc}}$ = $\\Large{latex(Rrc)}$\n'
    f'### Homogenous Transform between Camera  and Robot:\n'
    f'$\\large T_{{rc}}$ = $\\Large{latex(Trc)}$  \n'
    f'### Homogenous Transform between Robot and Camera:\n'
    f'$\\large T_{{cr}}  = \\large T_{{rc}}^{{-1}}$ = $\\Large{latex(Tcr)}$'
)

display(Markdown(string))


### Position between Camera  and Robot:
$\large p_{rc}$ = $\Large\left[\begin{matrix}0\\0\\0\end{matrix}\right]$
### Orientation between Camera  and Robot:
$\large R_{rc}$ = $\Large\left[\begin{matrix}0 & 0 & 1\\-1 & 0 & 0\\0 & -1 & 0\end{matrix}\right]$
### Homogenous Transform between Camera  and Robot:
$\large T_{rc}$ = $\Large\left[\begin{matrix}0 & 0 & 1 & 0\\-1 & 0 & 0 & 0\\0 & -1 & 0 & 0\\0 & 0 & 0 & 0\end{matrix}\right]$  
### Homogenous Transform between Robot and Camera:
$\large T_{cr}  = \large T_{rc}^{-1}$ = $\Large\left[\begin{matrix}0 & -1 & 0 & 0\\0 & 0 & -1 & 0\\1 & 0 & 0 & 0\\0 & 0 & 0 & 0\end{matrix}\right]$

## Visual Servoing Key Concepts

In [358]:
# Define the interaction matrix
Lx = sym.Matrix([
    [-(1/z),      0, (x / z),      x*y, -(1 + x**2), y],
    [0,      -(1/z), (y / z),   1+y**2, -(x * y),   -x]
])

Vc_sym_general = sym.Matrix([
    [sym.Symbol('v_x')],
    [sym.Symbol('v_y')],
    [sym.Symbol('v_z')],
    [sym.Symbol('w_x')],
    [sym.Symbol('w_y')],
    [sym.Symbol('w_z')]
])

string = (
    f'### Interaction Matrix: \n $\\large L_x$  = $\\large {latex(Lx)}$  \n'
    f'### Camera Velocity (optical frame): \n $\\Large V_c$ = $\\large{latex(Vc_sym_general)}$  \n'
    f'### Pixel velocity: \n $\\Large \dot{{s}}$ = $\\Large L_x \cdot \\Large V_c$ = $\\large {latex(Lx)} \cdot \large{latex(Vc_sym_general)}$'
)

display(Markdown(string))

### Interaction Matrix: 
 $\large L_x$  = $\large \left[\begin{matrix}- \frac{1}{Z{\left(t \right)}} & 0 & \frac{x{\left(t \right)}}{Z{\left(t \right)}} & x{\left(t \right)} y{\left(t \right)} & - x^{2}{\left(t \right)} - 1 & y{\left(t \right)}\\0 & - \frac{1}{Z{\left(t \right)}} & \frac{y{\left(t \right)}}{Z{\left(t \right)}} & y^{2}{\left(t \right)} + 1 & - x{\left(t \right)} y{\left(t \right)} & - x{\left(t \right)}\end{matrix}\right]$  
### Camera Velocity (optical frame): 
 $\Large V_c$ = $\large\left[\begin{matrix}v_{x}\\v_{y}\\v_{z}\\w_{x}\\w_{y}\\w_{z}\end{matrix}\right]$  
### Pixel velocity: 
 $\Large \dot{s}$ = $\Large L_x \cdot \Large V_c$ = $\large \left[\begin{matrix}- \frac{1}{Z{\left(t \right)}} & 0 & \frac{x{\left(t \right)}}{Z{\left(t \right)}} & x{\left(t \right)} y{\left(t \right)} & - x^{2}{\left(t \right)} - 1 & y{\left(t \right)}\\0 & - \frac{1}{Z{\left(t \right)}} & \frac{y{\left(t \right)}}{Z{\left(t \right)}} & y^{2}{\left(t \right)} + 1 & - x{\left(t \right)} y{\left(t \right)} & - x{\left(t \right)}\end{matrix}\right] \cdot \large\left[\begin{matrix}v_{x}\\v_{y}\\v_{z}\\w_{x}\\w_{y}\\w_{z}\end{matrix}\right]$

In [359]:
# Define the interaction matrix
Lx = sym.Matrix([
    [-(1/z),      0, (x / z),      x*y, -(1 + x**2), y],
    [0,      -(1/z), (y / z),   1+y**2, -(x * y),   -x]
])

Vc_sym = sym.Matrix([
    [sym.Symbol('0')],
    [sym.Symbol('0')],
    [sym.Symbol('v_z')],
    [sym.Symbol('0')],
    [sym.Symbol('w_y')],
    [sym.Symbol('0')]
])

Lx_inv = (Lx[:, [2, 4]])**-1 

twist = sym.Matrix([
    [sym.Symbol('v_z')],
    [sym.Symbol('w_y')],
])

string = (
    f'### In the case of a differential drive wheeled robot, the camera motion (camera optical frame) is allowed linearly along the z-axis and rotationally around the y-axis.  \n'
    f'#### Camera Velocity (optical frame): \n $\\Large V_c$ = $\\large {latex(Vc_sym)}$  \n'
    f'### Pixel velocity: \n $\\Large \dot{{s}}$ = $\\Large L_x \cdot \\large V_c$ = $\\Large{latex(Lx)} \cdot {latex(Vc_sym)} \Rightarrow$  \n'
    f'#### $ \Rightarrow \\large\dot{{s}} = {latex(Lx[:,[2, 4]])} \cdot {latex(Vc_sym[[2, 4],:])}$  \n'
    f'### So the velocity of the camera is:  \n '
    f'#### $\\Large V_c = \large L_x^{{-1}} \cdot \dot{{s}} \Rightarrow {latex(twist)} = { latex( Lx_inv )} \cdot \dot{{s}}$  \n'
    f'### However, this twist is expressed with respect to the camera\'s optical frame, and therefore it cannot be directly used as an input to the robot.'
)

display(Markdown(string))

### In the case of a differential drive wheeled robot, the camera motion (camera optical frame) is allowed linearly along the z-axis and rotationally around the y-axis.  
#### Camera Velocity (optical frame): 
 $\Large V_c$ = $\large \left[\begin{matrix}0\\0\\v_{z}\\0\\w_{y}\\0\end{matrix}\right]$  
### Pixel velocity: 
 $\Large \dot{s}$ = $\Large L_x \cdot \large V_c$ = $\Large\left[\begin{matrix}- \frac{1}{Z{\left(t \right)}} & 0 & \frac{x{\left(t \right)}}{Z{\left(t \right)}} & x{\left(t \right)} y{\left(t \right)} & - x^{2}{\left(t \right)} - 1 & y{\left(t \right)}\\0 & - \frac{1}{Z{\left(t \right)}} & \frac{y{\left(t \right)}}{Z{\left(t \right)}} & y^{2}{\left(t \right)} + 1 & - x{\left(t \right)} y{\left(t \right)} & - x{\left(t \right)}\end{matrix}\right] \cdot \left[\begin{matrix}0\\0\\v_{z}\\0\\w_{y}\\0\end{matrix}\right] \Rightarrow$  
#### $ \Rightarrow \large\dot{s} = \left[\begin{matrix}\frac{x{\left(t \right)}}{Z{\left(t \right)}} & - x^{2}{\left(t \right)} - 1\\\frac{y{\left(t \right)}}{Z{\left(t \right)}} & - x{\left(t \right)} y{\left(t \right)}\end{matrix}\right] \cdot \left[\begin{matrix}v_{z}\\w_{y}\end{matrix}\right]$  
### So the velocity of the camera is:  
 #### $\Large V_c = \large L_x^{-1} \cdot \dot{s} \Rightarrow \left[\begin{matrix}v_{z}\\w_{y}\end{matrix}\right] = \left[\begin{matrix}- Z{\left(t \right)} x{\left(t \right)} & \frac{Z{\left(t \right)} x^{2}{\left(t \right)}}{y{\left(t \right)}} + \frac{Z{\left(t \right)}}{y{\left(t \right)}}\\-1 & \frac{x{\left(t \right)}}{y{\left(t \right)}}\end{matrix}\right] \cdot \dot{s}$  
### However, this twist is expressed with respect to the camera's optical frame, and therefore it cannot be directly used as an input to the robot.

### Adjoint Transformation

For this reason, an adjoint transformation will be used to express the rotation with respect to the robot's frame, so that it can be provided as input to the robot.

In [360]:
def make_adjoint():
    pass

In [361]:
p1 = sym.Symbol('p1') 
p2 = sym.Symbol('p2') 
p3 = sym.Symbol('p3') 

p = sym.Matrix([
    [p1],
    [p2],
    [p3]
])

skew_p = sym.Matrix([
    [0, -p3, p2],
    [p3, 0, -p1],
    [-p2, p1, 0]
])

R = sym.Symbol('R') 
p_hat = sym.Symbol('phat')

Ad_T = sym.Matrix([
    [R, p_hat*R],
    [0,R]
])

Ad_Trc = sym.Matrix([
    [Rrc, skew_p * Rrc],
    [O3, Rrc] 
])

string = (
    f'### Skew symetric of position vector:  \n'
    f'#### $ \left[\!\left[ p \\right]\!\\right] $ $ή$ $\hat{{p}} = \\langle Skew \\rangle {latex(p)}  = {latex(skew_p)}$  '
    f' = ${latex(skew_p.subs({p1: Prc[0], p2: Prc[1], p3: Prc[2]}))}$   \n'
    f'### Adjoint Transformation for camera twist: \n'
    f'#### $ Ad_{{T_{{rc}}}} = {latex(Ad_T)} =  {latex(Ad_Trc)} \Rightarrow Ad_{{T_{{rc}}}} = {latex(Ad_Trc.subs({p1: Prc[0], p2: Prc[1], p3: Prc[2]}))}$'
)

display(Markdown(string))

### Skew symetric of position vector:  
#### $ \left[\!\left[ p \right]\!\right] $ $ή$ $\hat{p} = \langle Skew \rangle \left[\begin{matrix}p_{1}\\p_{2}\\p_{3}\end{matrix}\right]  = \left[\begin{matrix}0 & - p_{3} & p_{2}\\p_{3} & 0 & - p_{1}\\- p_{2} & p_{1} & 0\end{matrix}\right]$   = $\left[\begin{matrix}0 & 0 & 0\\0 & 0 & 0\\0 & 0 & 0\end{matrix}\right]$   
### Adjoint Transformation for camera twist: 
#### $ Ad_{T_{rc}} = \left[\begin{matrix}R & R \hat{p}\\0 & R\end{matrix}\right] =  \left[\begin{matrix}0 & 0 & 1 & p_{3} & - p_{2} & 0\\-1 & 0 & 0 & 0 & p_{1} & p_{3}\\0 & -1 & 0 & - p_{1} & 0 & - p_{2}\\0 & 0 & 0 & 0 & 0 & 1\\0 & 0 & 0 & -1 & 0 & 0\\0 & 0 & 0 & 0 & -1 & 0\end{matrix}\right] \Rightarrow Ad_{T_{rc}} = \left[\begin{matrix}0 & 0 & 1 & 0 & 0 & 0\\-1 & 0 & 0 & 0 & 0 & 0\\0 & -1 & 0 & 0 & 0 & 0\\0 & 0 & 0 & 0 & 0 & 1\\0 & 0 & 0 & -1 & 0 & 0\\0 & 0 & 0 & 0 & -1 & 0\end{matrix}\right]$

## Coming all together

In [373]:
Ad_Trc_subs = Ad_Trc.subs({p1: Prc[0], p2: Prc[1], p3: Prc[2]})
Vc_sym_short = Vc_sym[[2, 4],0]
Vcr = Ad_Trc_subs[:, [2, 4]]*Vc_sym_short

robot_twist = sym.Matrix([
    [sym.symbols('v_x')],
    [sym.symbols('w_z')]
])

Lx_final = Lx_inv.copy()
Lx_final[1, :] = -1 * Lx_inv[1, :]

string = (
    f'### Twist from the perspective of robot  \n'
    f'$\\Large V_c^r = \\Large Ad_{{T_{{rc}}}} \cdot \\Large V_c^c \Rightarrow$  \n\n'
    f'$\Rightarrow \\Large V_c^r = \\Large {latex(Ad_Trc_subs)} \cdot \\Large {latex(Vc_sym)}^c$  '
    f'$\Rightarrow \\Large V_c^r = \\Large {latex(Ad_Trc_subs[:, [2, 4]])} \cdot \\Large {latex(Vc_sym_short)}^c$  '
    f'$\Rightarrow \\Large V_c^r = {latex(Vc_sym_general)}^r = {latex(Vcr)}^c$ \n'
    f'### Finally  \n'
    f'$\\Large V_c^r = \\Large {latex(robot_twist)} = \\Large {latex(Lx_final)} \cdot \dot{{s}}$     \n'
)

display(Markdown(string))

### Twist from the perspective of robot  
$\Large V_c^r = \Large Ad_{T_{rc}} \cdot \Large V_c^c \Rightarrow$  

$\Rightarrow \Large V_c^r = \Large \left[\begin{matrix}0 & 0 & 1 & 0 & 0 & 0\\-1 & 0 & 0 & 0 & 0 & 0\\0 & -1 & 0 & 0 & 0 & 0\\0 & 0 & 0 & 0 & 0 & 1\\0 & 0 & 0 & -1 & 0 & 0\\0 & 0 & 0 & 0 & -1 & 0\end{matrix}\right] \cdot \Large \left[\begin{matrix}0\\0\\v_{z}\\0\\w_{y}\\0\end{matrix}\right]^c$  $\Rightarrow \Large V_c^r = \Large \left[\begin{matrix}1 & 0\\0 & 0\\0 & 0\\0 & 0\\0 & 0\\0 & -1\end{matrix}\right] \cdot \Large \left[\begin{matrix}v_{z}\\w_{y}\end{matrix}\right]^c$  $\Rightarrow \Large V_c^r = \left[\begin{matrix}v_{x}\\v_{y}\\v_{z}\\w_{x}\\w_{y}\\w_{z}\end{matrix}\right]^r = \left[\begin{matrix}v_{z}\\0\\0\\0\\0\\- w_{y}\end{matrix}\right]^c$ 
### Finally  
$\Large V_c^r = \Large \left[\begin{matrix}v_{x}\\w_{z}\end{matrix}\right] = \Large \left[\begin{matrix}- Z{\left(t \right)} x{\left(t \right)} & \frac{Z{\left(t \right)} x^{2}{\left(t \right)}}{y{\left(t \right)}} + \frac{Z{\left(t \right)}}{y{\left(t \right)}}\\1 & - \frac{x{\left(t \right)}}{y{\left(t \right)}}\end{matrix}\right] \cdot \dot{s}$     
